<a href="https://colab.research.google.com/github/netsetos/agentic-ai-weekend-gcp-learners/blob/main/module-08-agents-and-adk/lesson-8.1-root-agent/notebooks/GCP_Capstone_8.1_RootAgent.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 8.1 Root Agent with ADK — The Kit's Tool Layer, a Tenant the Model Cannot Choose, One Asserted Turn
**Netsetos GenAI Engineering — GCP Capstone** · Module 8 · rebuilt on the live lane, 8 September 2026

An ADK agent over the deployed lane. The tools are the kit's own (`deploy/shared/documind_tools.py`, imported from a clone, never pasted), the tenant comes from session state the runner seeds, the first turn is asserted to have called a tool, a callback answers instead of a tool, and Module 7's MCP server sits in the same tool list as the Python functions. The last cell writes an `adk web` package that imports the same kit.


## Setup


In [ ]:
!pip install -q "google-adk[mcp]==2.8.0" google-genai==2.22.0 google-auth==2.57.1 requests==2.34.2

from google.colab import auth
auth.authenticate_user()

PROJECT_ID = "documind-ai-YOUR-ID"   # CHANGE THIS: the project the lane runs in (make up, lesson 4.8)
REGION     = "us-central1"
TENANT     = "acme"
KIT        = "/content/agentic-ai-weekend-gcp-learners"   # the kit: deploy/shared is the tool layer every agent in this module imports
BRANCH     = "main"        # the learner repo's branch: the notebooks and the kit (deploy/) ship there together

import os, subprocess, sys
import google.auth
from google.auth.transport.requests import AuthorizedSession

if not os.path.isdir(KIT):
    subprocess.run(["git", "clone", "--depth", "1", "-q", "-b", BRANCH,
                    "https://github.com/netsetos/agentic-ai-weekend-gcp-learners", KIT], check=True)
sys.path.insert(0, f"{KIT}/deploy")                   # `from shared import ...` - the same layer every service imports

# The lane's URLs are deterministic: service name + project NUMBER (eventarc.tf builds them the same way).
creds, _ = google.auth.default()
NUMBER = AuthorizedSession(creds).get(
    f"https://cloudresourcemanager.googleapis.com/v1/projects/{PROJECT_ID}").json()["projectNumber"]
os.environ.update({
    "GOOGLE_CLOUD_PROJECT": PROJECT_ID,
    "GOOGLE_CLOUD_LOCATION": "global",              # Gemini 3.x generation is served from the global endpoint
    "GOOGLE_GENAI_USE_VERTEXAI": "TRUE",
    "DOCUMIND_PROFILE": "gcp",
    "RAG_API_URL": f"https://documind-api-{NUMBER}.{REGION}.run.app",
    "RAG_TIMEOUT_S": "90",                          # 7.2's finding: a cold API takes longer than the default 20 s
    # A notebook has no metadata server to be anyone with: the kit mints its ID tokens AS this roster
    # member (7.1). On Cloud Run the service's own account is the identity and nothing is set.
    "DOCUMIND_IMPERSONATE_SA": f"documind-ui-sa@{PROJECT_ID}.iam.gserviceaccount.com",
})
MCP_URL = f"https://documind-mcp-{NUMBER}.{REGION}.run.app"      # 7.2's server, for Cell 6

from shared import documind_tools    # THE one retrieve(). Imported, never pasted - 8.7's gate fails a paste.
print("kit:", KIT, "| API:", os.environ["RAG_API_URL"])


## Cell 1: The tools, adapted - not copied
The kit's `retrieve()` takes a `tenant_id`. An agent's tool must not: the model fills every parameter it can see. The adapter takes the tenant from session state and delegates. The probe at the end asks the lane one question with no model involved, so the first failure you can meet is a plain dict.


In [ ]:
from google.adk.tools import ToolContext

# TWO ADAPTERS, ZERO IMPLEMENTATIONS. The kit's retrieve() takes a tenant_id; an agent's tool must
# not, because the model fills every parameter it can see. So the adapter takes the tenant from
# SESSION STATE - seeded by the runner, never by the model - and delegates. That is the whole
# difference between an adapter and a copy: the body is one call, and there is no second place
# where "retrieval" is defined. deploy/services/chat/brains.py ships the same idea for the ADK
# brain through a before_tool_callback; Cell 4 uses that seam too.

def retrieve(query: str, doc_type: str = "all", top_k: int = 5, tool_context: ToolContext = None) -> dict:
    """Retrieve grounded passages from DocuMind's corpus, with the lane's own cited answer.

    Args:
        query: The question, in natural language.
        doc_type: policy, contract, invoice, report, statute, guidance, form, research_paper, or all.
        top_k: How many passages to return (1-20).
    """
    # tool_context is injected by ADK and absent from the schema the model reads. Its state was
    # seeded with the tenant when the session was created (Cell 3): the tenant is the runner's
    # decision here, exactly as it is the verified identity's in the chat service (12.8).
    state = tool_context.state if tool_context is not None else {}
    tenant = state.get("tenant_id") or TENANT
    if tool_context is not None:
        state["search_history"] = list(state.get("search_history", [])) + [query]
    return documind_tools.retrieve(query, tenant_id=tenant, top_k=max(1, min(int(top_k), 20)),
                                   doc_type=None if doc_type in ("all", "") else doc_type, brain="adk")


def calculate_cost(total_pages: int, processing_type: str = "standard") -> dict:
    """Estimate document processing cost in USD and INR.

    Args:
        total_pages: Total page count across all documents.
        processing_type: Service tier - standard, priority, or bulk.
    """
    try:
        return documind_tools.calculate_processing_cost(total_pages, processing_type=processing_type)
    except ValueError as e:
        # Data, not an exception. An exception ends the turn; a payload lets the model read which
        # tiers exist and ask again. Every tool in this course returns its errors.
        return {"error": str(e), "valid_tiers": sorted(documind_tools.RATES)}


# One question straight through the adapter, before any model is involved. If this fails, nothing
# below can work, and the failure is a plain dict you can read: no agent to blame yet.
probe = retrieve("After how many years of continuous service does gratuity become payable?")
assert probe.get("answerable") and probe.get("citations"), probe
print("the lane answers:", (probe.get("answer") or "")[:90], "| citations:", len(probe["citations"]))


## Cell 2: The agent
The instruction says when to call which tool and what never to do. There is no `generate_content_config`: gemini-3.6-flash ignores sampling parameters.


In [ ]:
from google.adk.agents import LlmAgent

INSTRUCTION = (
    "You are DocuMind AI, answering questions about the company's documents. "
    "For any question about the documents call retrieve, answer only from what it returns and cite "
    "the sources it names. Use calculate_cost to price a set of pages. "
    "If a tool says the corpus cannot answer, say so and cite nothing. Never state a figure a citation does not carry."
)

root_agent = LlmAgent(
    name="documind",
    model="gemini-3.6-flash",
    instruction=INSTRUCTION,
    # No generate_content_config: gemini-3.6-flash ignores temperature, top_p and top_k. Setting one
    # buys nothing but the belief that you tuned something.
    tools=[retrieve, calculate_cost],
)
print("agent:", root_agent.name, "| tools:", [t.__name__ for t in (retrieve, calculate_cost)])


## Cell 3: One turn, one tool call, one cited answer
The tenant is seeded into the session by the runner. The assert is the gate: an agent that answered without a tool answered from memory.


In [ ]:
from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService
from google.genai import types

LAST = {"calls": [], "results": [], "text": ""}
session_service = InMemorySessionService()
runner = Runner(agent=root_agent, app_name="documind", session_service=session_service)
# The tenant is seeded HERE, by the runner, from something the model never touches. In the chat
# service that something is the person's verified identity and the roster (12.8).
session = await session_service.create_session(app_name="documind", user_id="student", state={"tenant_id": TENANT})


async def run_turn(question: str, target=None, session_id=None) -> bool:
    """Run one turn; print every tool call, tool result and text part. Returns whether a tool was called."""
    LAST.update(calls=[], results=[], text="")
    r = runner if target is None else Runner(agent=target, app_name="documind", session_service=session_service)
    content = types.Content(role="user", parts=[types.Part.from_text(text=question)])
    async for event in r.run_async(user_id="student", session_id=session_id or session.id, new_message=content):
        for part in (event.content.parts if event.content and event.content.parts else []):
            if part.function_call:
                LAST["calls"].append(part.function_call.name)
                print(f"  [tool call] {part.function_call.name}({dict(part.function_call.args or {})})")
            if part.function_response:
                LAST["results"].append(part.function_response.response)
                print(f"  [tool result] {str(part.function_response.response)[:160]}")
            if part.text:
                LAST["text"] += part.text
                print(f"  [agent] {part.text.strip()[:400]}")
    return bool(LAST["calls"])


# THIS RUNS, AND IT IS ASSERTED. An agent that answers without calling a tool has answered from
# memory - the failure this module exists to catch - and a lesson whose last cell is commented out
# never catches it.
called = await run_turn("What is the notice period for a confirmed E3?")          # golden row lk-06: 60 days
assert called, "no tool call - the model answered from memory (is the lane deployed? may your account mint as ui-sa?)"
assert any(isinstance(r, dict) and r.get("citations") for r in LAST["results"]), "the tool returned no citations"

# A follow-up. The SESSION holds the conversation, so "probation" resolves against the last question.
await run_turn("And during probation?")                                            # lk-04: 15 days


## Cell 4: The guard answers
`before_tool_callback` runs between the model's decision and the tool's execution. Return a dict and the model reads it as the tool's result; return `None` and the call goes through. `after_tool_callback` sees the result on the way back.


In [ ]:
# THE GUARD ANSWERS INSTEAD OF THE TOOL. before_tool_callback runs between the model's decision
# and the tool's execution: return None to let the call through (with args rewritten if you like),
# or a dict to REPLACE the call - the model reads your dict as if the tool had returned it. It is
# the seam the chat service uses to bind the tenant and refuse blocked tools (brains.py, 8.7), and
# 6.4's wrap_tool_call and 8.6's confirm node are the same seam in other clothes: decide, then act.
def guard(tool, args, tool_context):
    if tool.name == "calculate_cost" and args.get("processing_type") not in documind_tools.RATES:
        return {"error": f"unknown tier {args.get('processing_type')!r}", "valid_tiers": sorted(documind_tools.RATES)}
    if tool.name == "retrieve" and not str(args.get("query", "")).strip():
        return {"error": "empty query", "citations": [], "answerable": False}
    return None


def remember(tool, args, tool_context, tool_response):
    """after_tool_callback: None keeps the tool's result, a dict replaces it. Here it only records."""
    tool_context.state["last_tool"] = tool.name
    return None


guarded = LlmAgent(name="documind_guarded", model="gemini-3.6-flash", instruction=INSTRUCTION,
                   tools=[retrieve, calculate_cost], before_tool_callback=guard, after_tool_callback=remember)

# "premium" is not a tier. The guard answers, the model reads the valid tiers and asks again - look
# for TWO calculate_cost calls in the trace: one replaced by the guard, one priced by the tool.
await run_turn("What would it cost to process 250 pages at the premium tier? "
               "If that tier does not exist, use the closest real one.", target=guarded)
print("\ncalls this turn:", LAST["calls"])


## Cell 5: State across turns


In [ ]:
s = await session_service.get_session(app_name="documind", user_id="student", session_id=session.id)
print("state :", dict(s.state))
print("events:", len(s.events), "| last question the corpus was asked:", s.state.get("search_history", [])[-1:])

# What state holds and what it does not. The tenant (the runner seeded it), the search history (the
# adapter wrote it), the last tool (the callback wrote it). NOT the documents and NOT the answers:
# those are EVENTS, and the conversation the model sees is rebuilt from events on every turn. State
# is for what the tools need to know across turns. 8.5 draws the same line between a checkpoint
# (the conversation) and a store (what you know about the user).


## Cell 6: Python tools and MCP tools in one list
Module 7's server, from inside the kit: the same `McpToolset` and per-call credential 7.3 taught, filtered to the two tools the Python side does not have, under a prefix.


In [ ]:
from google.adk.tools.mcp_tool import McpToolset
from google.adk.tools.mcp_tool.mcp_session_manager import StreamableHTTPConnectionParams

# PYTHON TOOLS AND MCP TOOLS IN ONE LIST. Module 7's server exposes the lane's operations to agents
# outside the kit; an agent inside the kit can still borrow them - here for what the Python side has
# no tool for, corpus counts and the document list. The model sees one flat list of names and
# descriptions; it does not know or care which are in-process and which are a network hop. The
# PREFIX is what keeps two servers' `search` from shadowing each other, so it is not decoration.
lane = McpToolset(
    # timeout=120: ADK's default is 5 s, and a retrieve() through the server is rag-api plus a Gemini
    # answer - up to 90 s when the API is cold. A roster refusal is instant, which is how a 5 s client
    # passes the refusal test and fails the real question (the first live A2A peer did exactly that).
    connection_params=StreamableHTTPConnectionParams(url=f"{MCP_URL}/mcp", timeout=120, sse_read_timeout=300),
    # Minted on EVERY tool call (7.3): the kit's own hook, as the roster member this notebook speaks as.
    header_provider=lambda ctx: {"Authorization": f"Bearer {documind_tools._id_token(MCP_URL)}"},
    tool_filter=["corpus_stats", "list_documents"],
    tool_name_prefix="mcp",
)

mixed = LlmAgent(
    name="documind_mixed", model="gemini-3.6-flash",
    instruction=INSTRUCTION + " Use mcp_corpus_stats for how many documents or chunks the corpus holds and "
                              "mcp_list_documents to see what is indexed; pass tenant='acme' to both.",
    tools=[retrieve, calculate_cost, lane],
)
await run_turn("How many chunks does our corpus hold, and which documents are indexed?", target=mixed)
print("\ncalls this turn:", LAST["calls"])


## Cell 7: An `adk web` package that imports the kit


In [ ]:
# adk web wants a PACKAGE with a root_agent. The package imports the kit the way this notebook does -
# nothing pasted - and .env carries only names and addresses. No key, no token: the identity is
# minted per call from the credential the shell already has.
import textwrap

os.makedirs("documind_agent", exist_ok=True)
open("documind_agent/__init__.py", "w").write("from . import agent\n")
open("documind_agent/agent.py", "w").write(textwrap.dedent(f'''
    import os, sys
    sys.path.insert(0, os.environ.get("DOCUMIND_KIT", {KIT!r}) + "/deploy")
    from shared import documind_tools
    from google.adk.agents import LlmAgent
    from google.adk.tools import ToolContext

    TENANT = os.environ.get("TENANT", {TENANT!r})


    def retrieve(query: str, doc_type: str = "all", top_k: int = 5, tool_context: ToolContext = None) -> dict:
        """Retrieve grounded passages from DocuMind's corpus, with the lane's own cited answer.

        Args:
            query: The question, in natural language.
            doc_type: policy, contract, invoice, report, statute, guidance, form, research_paper, or all.
            top_k: How many passages to return (1-20).
        """
        state = tool_context.state if tool_context is not None else {{}}
        return documind_tools.retrieve(query, tenant_id=state.get("tenant_id") or TENANT, top_k=max(1, min(int(top_k), 20)),
                                       doc_type=None if doc_type in ("all", "") else doc_type, brain="adk")


    def calculate_cost(total_pages: int, processing_type: str = "standard") -> dict:
        """Estimate document processing cost in USD and INR.

        Args:
            total_pages: Total page count across all documents.
            processing_type: Service tier - standard, priority, or bulk.
        """
        try:
            return documind_tools.calculate_processing_cost(total_pages, processing_type=processing_type)
        except ValueError as e:
            return {{"error": str(e), "valid_tiers": sorted(documind_tools.RATES)}}


    root_agent = LlmAgent(name="documind", model="gemini-3.6-flash", instruction={INSTRUCTION!r},
                          tools=[retrieve, calculate_cost])
''').lstrip())
open("documind_agent/.env", "w").write("\n".join(
    f"{k}={os.environ[k]}" for k in ("GOOGLE_GENAI_USE_VERTEXAI", "GOOGLE_CLOUD_PROJECT", "GOOGLE_CLOUD_LOCATION",
                                     "DOCUMIND_PROFILE", "RAG_API_URL", "RAG_TIMEOUT_S", "DOCUMIND_IMPERSONATE_SA")) + "\n")
print("package written: documind_agent/. In a shell with the same credential: `adk web` here, then pick documind_agent.")


## Where this goes
- **8.2** builds teams of these agents: routing, pipelines, fan-out across two tenants, a refinement loop.
- **8.7** puts the same tool under three harnesses and prints three cost lines; `deploy/services/chat/brains.py` is this agent shipped, with the tenant bound from the verified identity in a `before_tool_callback`.

## ✅ Lesson 8.1 complete
- ✅ The kit's tool layer imported from a clone; two adapters, zero implementations
- ✅ The tenant from session state the runner seeded - never a model parameter
- ✅ One turn asserted to have called a tool and returned citations; a follow-up resolved by the session
- ✅ A guard that answers instead of a tool; a callback that records
- ✅ MCP tools beside Python tools, one credential per call, one prefix
- ✅ An `adk web` package that imports the same kit
